# Constant Acceleration MPC Testing

In [ ]:
%matplotlib ipympl

import jax

jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
# jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
# jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
# jax.config.update("jax_disable_jit", True)

In [ ]:
import functools

import jax.numpy as jnp
import matplotlib.pyplot as plt  # leave for convenient debugging
import numpy as np
import tqdm

from exp_mpc.stewart_min import mp_mpl, mpc_spec, opt, viz

In [ ]:
# setup
# weights = mpc_spec.ExpWeights(
#     lin_dyn=jnp.ones(3) * 1e5,
#     omega=jnp.ones(3) * 5e5,
#     control=jnp.ones(6) * 1e-1,
#     alpha_acc=jnp.array([1.0]),
#     alpha_omega=jnp.array([1.0]),
# )
weights = mpc_spec.ExpWeights(
    lin_dyn=jnp.ones(3) * 1e4,
    omega=jnp.ones(3) * 1e4,
    control=jnp.ones(6) * 1e-1,
    alpha_acc=jnp.array([0.0]),
    alpha_omega=jnp.array([0.0]),
)
limits = mpc_spec.MPCLimits()
spec = mpc_spec.MPCSpec.init_weight_margins(weights, limits, max_iter=2, max_ls=1, use_terminal=True, debug=False, init_norm=1e-1)
train_step = functools.partial(opt.train_step_with_cost, spec=spec, use_scipy=False)

T = 20.0  # s
num_steps = int(T / spec.dt)
n = 200  # horizon

gravity = mpc_spec.gravity
acc_ref = jnp.array([1.0, 0.0, 0.0]) + gravity
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([0.0, 0.0, 0.1])
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

gravity_ref = jnp.tile(A=gravity, reps=(n, 1))
zero_ref = jnp.zeros((n, 3))

In [ ]:
# run setup
train_state = opt.TrainState.zero_init(spec, n)
train_state.vstate0_sim = train_state.vstate0_irl  # sim is on earth
train_list = [train_state]
times = []
res_list = []

In [ ]:
# precompile
print(train_step(acc_ref, omega_ref, train_state)[-1])  # compile + run-time

In [ ]:
# run;
for i in tqdm.tqdm(range(num_steps)):
# for i in tqdm.tqdm(range(2**8)):
    if i <= num_steps:
        aref = acc_ref
        oref = omega_ref
    else:
        aref = gravity_ref
        oref = zero_ref
    train_state, res, t_tot = train_step(aref, oref, train_state)
    train_list.append(train_state)
    res_list.append(res)
    times.append(t_tot)

In [ ]:
freqs = 1.0 / np.array(times)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
tl = train_list
references = {
     "xyz-acceleration": jnp.tile(A=acc_ref[0], reps=(len(tl), 1)),
     "angular-velocity": jnp.tile(A=omega_ref[0], reps=(len(tl), 1)),
}

In [ ]:
mpc_human_fig = viz.plot_human_trajectory(trajectory=tl, limits=limits, spec=spec, references=references)

In [ ]:
mpc_vestibular_fig = viz.plot_vestibular_trajectory(trajectory=tl, limits=limits, spec=spec)

In [ ]:
mpc_table_fig = viz.plot_cartesian_table_trajectory(trajectory=tl, limits=limits, spec=spec)

In [ ]:
mpc_actuator_fig = viz.plot_actuator_trajectory(trajectory=tl, limits=limits, spec=spec, use_estop=True)

In [ ]:
assert False

## animations

(WARNING: can take a long time.
Usually about as long as the video being generated.)

In [ ]:
mp_mpl.call_mp_animate_trajectory(
    file_name="../data/const_acc_3d.mp4",
    trajectory=train_list,
    limits=limits,
    spec=spec,
)